# Image to LaTeX -- EASY TO USE BATTERY INCLUDED :)

## Two methods, one pipeline

We solve image -> LaTeX two ways and compare them:

- **Method 1 -- RNN:** CNN + biLSTM encoder + LSTM decoder with attention
- **Method 2 -- Transformer:** CNN + Transformer encoder + Transformer decoder

Both share the exact same data, vocabulary and pipeline -- only the sequence
model changes, so the comparison is fair. To pick a method, set `MODEL_TYPE`
in `config.py` to `"rnn"` or `"transformer"`, then run the cells below
(train -> predict -> evaluate).

Everything that differs between the two runs is saved under a per-method name --
checkpoints (`model_best_rnn.pt` / `model_best_transformer.pt`) and prediction
files (`test_formulas_rnn.txt` / `test_formulas_transformer.txt`, and likewise for
`val_predictions_*`) -- so running one method never overwrites the other. Run the
notebook once for each method and copy the printed BLEU / Edit-Distance /
Exact-Match numbers into the results table in the README.

## 1. Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -r /content/drive/MyDrive/visual-latex-decompiler/requirements.txt -q

## 2. Load Project From google drive


In [2]:
# === 2. RUN AFTER EVERY RUNTIME RESET ===

!mkdir -p /content/visual-latex-decompiler

# Copy .py files (tiny, instant)
!cp /content/drive/MyDrive/visual-latex-decompiler/*.py /content/visual-latex-decompiler/

# Copy zip + extract to local SSD (fast — one big file, -o = overwrite without asking)
!cp /content/drive/MyDrive/visual-latex-decompiler/Data_Im2Latx.zip /content/
!unzip -qo /content/Data_Im2Latx.zip -d /content/visual-latex-decompiler/
!rm /content/Data_Im2Latx.zip

# Restore checkpoints from Drive if they exist (resume training)
import os, shutil
drive_ckpt = '/content/drive/MyDrive/visual-latex-decompiler/checkpoints'
local_ckpt = '/content/visual-latex-decompiler/checkpoints'
if os.path.exists(drive_ckpt):
    os.makedirs(local_ckpt, exist_ok=True)
    for f in os.listdir(drive_ckpt):
        shutil.copy2(os.path.join(drive_ckpt, f), local_ckpt)
    print('Restored checkpoints from Drive!')
    !ls -la {local_ckpt}
else:
    print('No checkpoints found on Drive at:', drive_ckpt)

# Also copy vocab.pkl if it exists (avoids rebuilding)
drive_vocab = '/content/drive/MyDrive/visual-latex-decompiler/vocab.pkl'
if os.path.exists(drive_vocab):
    shutil.copy2(drive_vocab, '/content/visual-latex-decompiler/')
    print('Restored vocab.pkl from Drive!')

%cd /content/visual-latex-decompiler
!ls

## 3. Check GPU

In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Check Dataset

In [4]:
import os
data_dir = "Data_Im2Latx"
print("Dataset contents:")
for item in os.listdir(data_dir):
    full = os.path.join(data_dir, item)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print("  {}/  ({} files)".format(item, n))
    else:
        sz = os.path.getsize(full)
        print("  {}  ({:.1f} KB)".format(item, sz/1024))

## 5. Build Vocabulary

In [ ]:
import os

# rebuild if it's missing, so we don't overwrite the uploaded one.
if os.path.exists("vocab.pkl"):
    print("Using existing vocab.pkl (uploaded from Drive) -- skipping rebuild.")
else:
    print("No vocab.pkl found -- building vocabulary...")
    !python vocab.py

## 6. Train the selected model
Trains whichever method `MODEL_TYPE` points to in `config.py`
(`"rnn"` or `"transformer"`).

In [ ]:
!python train.py

# Auto-save checkpoints to Drive after training
import shutil, os
save_dir = '/content/drive/MyDrive/visual-latex-decompiler/checkpoints'
os.makedirs(save_dir, exist_ok=True)
if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', save_dir, dirs_exist_ok=True)
    print('Checkpoints saved to Drive!')

## 7. View Training Plots

In [ ]:
from IPython.display import Image, display
import config as C

# per-method plot files (loss_rnn.png / loss_transformer.png etc.)
m = C.MODEL_TYPE
print("--- Loss ({}) ---".format(m))
display(Image("plots/loss_{}.png".format(m)))
print("--- BLEU ({}) ---".format(m))
display(Image("plots/bleu_{}.png".format(m)))
print("--- Edit Distance ({}) ---".format(m))
display(Image("plots/edit_distance_{}.png".format(m)))

## 8. Generate Test Predictions (beam search + post-processing)

In [ ]:
!python predict.py --beam --postprocess

## 9. Preview Predictions

In [ ]:
import config as C
mt = C.MODEL_TYPE
pred_file = "test_formulas_{}.txt".format(mt)
with open(pred_file, "r") as f:
    lines = f.readlines()
print("Method:", mt, "| file:", pred_file)
print("Total predictions:", len(lines))
print("\nFirst 10:")
for i, line in enumerate(lines[:10]):
    print("  [{}] {}".format(i, line.strip()))

## 10. Evaluate Test Predictions (BLEU + Edit Distance + Exact Match)
Checks if test ground truth exists; if not, evaluates on the validation set
instead. Runs all three metrics for the selected method.

In [ ]:
import os
import config as C

mt = C.MODEL_TYPE
test_gt = "Data_Im2Latx/test_formulas.txt"
predicted = "test_formulas_{}.txt".format(mt)

if os.path.exists(test_gt):
    print("=== Test Set BLEU (4-gram) [{}] ===".format(mt))
    !python bleu_score.py --target-formulas {test_gt} --predicted-formulas {predicted} --ngram 4

    print("\n=== Test Set Edit Distance [{}] ===".format(mt))
    !python edit_distance.py --target-formulas {test_gt} --predicted-formulas {predicted}

    print("\n=== Test Set Exact Match [{}] ===".format(mt))
    !python exact_match.py --target-formulas {test_gt} --predicted-formulas {predicted}
else:
    print("No test ground truth found at:", test_gt)
    print("Skipping test evaluation. Will evaluate on validation set instead.")

    val_pred = "val_predictions_{}.txt".format(mt)
    if os.path.exists(val_pred):
        print("\n=== Validation Set BLEU (4-gram) [{}] ===".format(mt))
        !python bleu_score.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas {val_pred} --ngram 4
        print("\n=== Validation Set Edit Distance [{}] ===".format(mt))
        !python edit_distance.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas {val_pred}
        print("\n=== Validation Set Exact Match [{}] ===".format(mt))
        !python exact_match.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas {val_pred}
    else:
        print(val_pred, "not generated yet. Run the validation cell first, then re-run this cell.")

## 11. Evaluate on Validation Set
Generates validation predictions with the selected model (beam + post-process),
then scores BLEU / Edit Distance / Exact Match.

In [ ]:
# first generate val predictions (beam search + postprocessing, same as test)
import config as C
import torch
from torch.utils.data import DataLoader
from vocab import Vocab
from dataset import FormulaDataset, collate_train
from model import build_model
from postprocess import clean_latex

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = Vocab.load()
# load whichever method config.MODEL_TYPE selects -- checkpoints are per method
ckpt_path = "checkpoints/model_best_{}.pt".format(C.MODEL_TYPE)
ckpt = torch.load(ckpt_path, map_location=dev)
model = build_model(ckpt["vocab_size"]).to(dev)
model.load_state_dict(ckpt["model"])
model.eval()
print("Loaded {} model from {}".format(C.MODEL_TYPE, ckpt_path))

val_ds = FormulaDataset(
    C.VAL_IMAGES_DIR, C.VAL_FORMULAS, vocab, training=False)
val_ld = DataLoader(val_ds, C.BATCH, shuffle=False, collate_fn=collate_train)

val_preds = []
done = 0
total = len(val_ds)
for imgs, tgts, lens in val_ld:
    done += imgs.size(0)
    print("\r  {}/{} images ({:.0f}%)".format(done,
          total, 100*done/total), end="", flush=True)
    imgs = imgs.to(dev)
    seqs = model.beam_decode(imgs, vocab.sos_id, vocab.eos_id, beam_k=C.BEAM_K)
    for seq in seqs:
        formula = " ".join(vocab.decode(seq))
        formula = clean_latex(formula)
        val_preds.append(formula)

print()
out_file = "val_predictions_{}.txt".format(C.MODEL_TYPE)
with open(out_file, "w") as f:
    for p in val_preds:
        f.write(p + "\n")
print("Wrote {} val predictions -> {}".format(len(val_preds), out_file))

In [ ]:
import config as C
mt = C.MODEL_TYPE
val_pred = "val_predictions_{}.txt".format(mt)

print("=== BLEU [{}] ===".format(mt))
!python bleu_score.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas {val_pred} --ngram 4

print("\n=== Edit Distance [{}] ===".format(mt))
!python edit_distance.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas {val_pred}

print("\n=== Exact Match [{}] ===".format(mt))
!python exact_match.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas {val_pred}

## 12. Visual Comparison: Image vs Prediction vs Ground Truth
Shows 8 random validation samples side by side. Each row: the input image, the ground truth LaTeX, and the model's prediction.

In [ ]:
import random
import difflib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.offsetbox import HPacker, TextArea, AnchoredOffsetbox, VPacker
from PIL import Image as PILImage
import config as C
from vocab import read_formulas

# Load ground truth and predictions (per-method file)
gt_formulas = read_formulas(C.VAL_FORMULAS)
val_pred_file = "val_predictions_{}.txt".format(C.MODEL_TYPE)
with open(val_pred_file, "r") as f:
    pred_formulas = [line.strip().split() for line in f.readlines()]

# Pick 8 random samples
num_samples = 8
indices = random.sample(range(len(gt_formulas)), num_samples)

fig = plt.figure(figsize=(16, num_samples * 2.8))
gs = gridspec.GridSpec(num_samples, 2, width_ratios=[
                       1, 2], hspace=0.5, wspace=0.3)


def build_diff_boxes(gt_tokens, pred_tokens, fontsize=6.5):
    """Return (gt_boxes, pred_boxes) with matching tokens green, diffs red."""
    sm = difflib.SequenceMatcher(None, gt_tokens, pred_tokens)
    gt_boxes = []
    pred_boxes = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == "equal":
            for t in gt_tokens[i1:i2]:
                gt_boxes.append(TextArea(
                    t + " ", textprops=dict(fontsize=fontsize, fontfamily="monospace", color="green")))
            for t in pred_tokens[j1:j2]:
                pred_boxes.append(TextArea(
                    t + " ", textprops=dict(fontsize=fontsize, fontfamily="monospace", color="green")))
        elif op == "replace":
            for t in gt_tokens[i1:i2]:
                gt_boxes.append(TextArea(t + " ", textprops=dict(fontsize=fontsize,
                                fontfamily="monospace", color="red", fontweight="bold")))
            for t in pred_tokens[j1:j2]:
                pred_boxes.append(TextArea(t + " ", textprops=dict(fontsize=fontsize,
                                  fontfamily="monospace", color="red", fontweight="bold")))
        elif op == "delete":
            for t in gt_tokens[i1:i2]:
                gt_boxes.append(TextArea(t + " ", textprops=dict(fontsize=fontsize,
                                fontfamily="monospace", color="red", fontweight="bold")))
        elif op == "insert":
            for t in pred_tokens[j1:j2]:
                pred_boxes.append(TextArea(t + " ", textprops=dict(fontsize=fontsize,
                                  fontfamily="monospace", color="red", fontweight="bold")))
    return gt_boxes, pred_boxes


for row, idx in enumerate(indices):
    img_path = os.path.join(C.VAL_IMAGES_DIR, "{:05d}.png".format(idx))
    img = PILImage.open(img_path).convert("L")

    gt_tokens = gt_formulas[idx]
    pred_tokens = pred_formulas[idx] if idx < len(pred_formulas) else ["N/A"]
    match = (gt_tokens == pred_tokens)

    # Left: image
    ax_img = fig.add_subplot(gs[row, 0])
    ax_img.imshow(img, cmap="gray")
    ax_img.set_title("Sample #{}".format(idx), fontsize=9, fontweight="bold")
    ax_img.axis("off")

    # Right: colored diff text
    ax_txt = fig.add_subplot(gs[row, 1])
    ax_txt.axis("off")
    ax_txt.set_xlim(0, 1)
    ax_txt.set_ylim(0, 1)

    if match:
        ax_txt.text(0.02, 0.72, "GT:   " + " ".join(gt_tokens)[:120],
                    fontsize=7, fontfamily="monospace", color="green", va="top")
        ax_txt.text(0.02, 0.32, "Pred: " + " ".join(pred_tokens)[:120],
                    fontsize=7, fontfamily="monospace", color="green", va="top")
        ax_txt.text(0.98, 0.5, "MATCH", fontsize=8, fontweight="bold",
                    color="green", ha="right", va="center")
        ax_txt.set_facecolor("#e8f5e9")
    else:
        # Truncate for display
        gt_trunc = gt_tokens[:30]
        pred_trunc = pred_tokens[:30]

        gt_boxes, pred_boxes = build_diff_boxes(gt_trunc, pred_trunc)

        # GT label + tokens
        gt_label = TextArea("GT:   ", textprops=dict(
            fontsize=7, fontfamily="monospace", color="black", fontweight="bold"))
        gt_row = HPacker(children=[gt_label] + gt_boxes,
                         pad=0, sep=0, align="baseline")

        # Pred label + tokens
        pred_label = TextArea("Pred: ", textprops=dict(
            fontsize=7, fontfamily="monospace", color="black", fontweight="bold"))
        pred_row = HPacker(children=[pred_label] +
                           pred_boxes, pad=0, sep=0, align="baseline")

        vbox = VPacker(children=[gt_row, pred_row], pad=2, sep=6, align="left")
        anchored = AnchoredOffsetbox(loc="center left", child=vbox, pad=0.2,
                                     bbox_to_anchor=(0.01, 0.5),
                                     bbox_transform=ax_txt.transAxes, frameon=False)
        ax_txt.add_artist(anchored)

        ax_txt.text(0.98, 0.5, "DIFF", fontsize=8, fontweight="bold",
                    color="red", ha="right", va="center")
        ax_txt.set_facecolor("#fff3e0")

plt.suptitle("Validation Samples ({}): Ground Truth vs Prediction (green=match, red=diff)".format(C.MODEL_TYPE),
             fontsize=12, fontweight="bold")
# per-method file so rnn and transformer runs don't clobber each other
comparison_file = "plots/comparison_{}.png".format(C.MODEL_TYPE)
plt.savefig(comparison_file, dpi=150, bbox_inches="tight")
plt.show()
print("Saved comparison plot to", comparison_file)

## 13. Save Results to Google Drive

In [ ]:
import shutil, os, glob

save_dir = "/content/drive/MyDrive/visual-latex-decompiler"
os.makedirs(save_dir, exist_ok=True)

# copy whatever prediction files exist (per-method names)
for f in glob.glob("test_formulas_*.txt") + glob.glob("val_predictions_*.txt"):
    shutil.copy(f, save_dir)
shutil.copytree("plots", os.path.join(save_dir, "plots"), dirs_exist_ok=True)
shutil.copytree("checkpoints", os.path.join(save_dir, "checkpoints"), dirs_exist_ok=True)
shutil.copytree("logs", os.path.join(save_dir, "logs"), dirs_exist_ok=True)

print("Saved everything to", save_dir)
print("Contents:")
for root, dirs, files in os.walk(save_dir):
    for f in files:
        print("  ", os.path.join(root, f))